In [1]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
train = pd.read_csv(
    "22121477/train_lending_club.csv"
)

test = pd.read_csv(
    "22121477/test_lending_club.csv"
)

In [3]:
features = [
    "int_rate",
    "fico_range_low",
    "dti",
    "annual_inc",
    "revol_util",
    "total_acc",
    "open_acc",
    "pub_rec",
    "pub_rec_bankruptcies",
    "mort_acc",
    "emp_length"
]

target = "loan_status"

loan_status_labeled = test["loan_status"].notna()

X_train = train[features]
y_train = train[target]
X_test = test.loc[loan_status_labeled, features]
y_test = test.loc[loan_status_labeled, target]

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
# validate data
print(X_train.dtypes)
print(X_train.isna().sum())
print(X_test.isna().sum())

(236846, 11)
(236846,)
(93607, 11)
(93607,)
int_rate                float64
fico_range_low          float64
dti                     float64
annual_inc              float64
revol_util              float64
total_acc               float64
open_acc                float64
pub_rec                 float64
pub_rec_bankruptcies    float64
mort_acc                float64
emp_length              float64
dtype: object
int_rate                0
fico_range_low          0
dti                     0
annual_inc              0
revol_util              0
total_acc               0
open_acc                0
pub_rec                 0
pub_rec_bankruptcies    0
mort_acc                0
emp_length              0
dtype: int64
int_rate                0
fico_range_low          0
dti                     0
annual_inc              0
revol_util              0
total_acc               0
open_acc                0
pub_rec                 0
pub_rec_bankruptcies    0
mort_acc                0
emp_length              0
dtype

In [4]:
model_pipeline = Pipeline([
    (
        "preprocessing_step",
        StandardScaler()
    ),
    (
        "model_step",
        LogisticRegression(
            max_iter= 1000
        )
    )
])

model_pipeline.fit(
    X_train,
    y_train
)

predicted_classes = model_pipeline.predict(
    X_test
)

predicted_probabilities = model_pipeline.predict_proba(
    X_test
)

model_pipeline.classes_

array([0, 1])

In [5]:
print(np.shape(predicted_classes))

print(np.shape(predicted_probabilities))

print(predicted_probabilities[:5])





(93607,)
(93607, 2)
[[0.01052288 0.98947712]
 [0.18135237 0.81864763]
 [0.29209136 0.70790864]
 [0.03347861 0.96652139]
 [0.19227155 0.80772845]]


In [14]:
print(f"Expanded model predicted success(1) vs defaults (0): \n{pd.Series(predicted_classes).value_counts()}\n")
print(f"Actual success(1) vs defaults (0): \n{pd.Series(y_test).value_counts()}")

Expanded model predicted success(1) vs defaults (0): 
1    85091
0     8516
Name: count, dtype: int64

Actual success(1) vs defaults (0): 
loan_status
1.0    78859
0.0    14748
Name: count, dtype: int64


In [7]:
cm = confusion_matrix(
    y_test,
    predicted_classes,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual default (0)", "Actual success (1)"],
    columns=["Predicted default (0)", "Predicted success (1)"]
)

cm_df

,Predicted default (0),Predicted success (1)
Actual default (0),4859,9889
Actual success (1),3657,75202


In [8]:
accuracy = accuracy_score(
    y_test,
    predicted_classes
)

default_precision = precision_score(
    y_test,
    predicted_classes,
    pos_label=0
)

default_recall = recall_score(
    y_test,
    predicted_classes,
    pos_label=0
)

default_f1 = f1_score(
    y_test,
    predicted_classes,
    pos_label=0
)

print("Accuracy:", accuracy)
print("Default precision:", default_precision)
print("Default recall:", default_recall)
print("Default F1-score:", default_f1)

Accuracy: 0.8552886002115226
Default precision: 0.5705730389854392
Default recall: 0.3294684024952536
Default F1-score: 0.4177269601100413


In [9]:
default_probabilities = predicted_probabilities[:,0]

average_probability_for_defaults = default_probabilities[y_test == 0].mean()

average_probability_for_successes = default_probabilities[y_test == 1].mean()

print(f"Average predicted default probability for actual defaults: {100 * average_probability_for_defaults:.2f}%")

print(f"Average predicted default probability for actual successes: {100 * average_probability_for_successes:.2f}%")

Average predicted default probability for actual defaults: 39.88%
Average predicted default probability for actual successes: 15.85%


In [10]:
print("Actual defaults:")
print(
    np.percentile(
        default_probabilities[y_test == 0],
        [25, 50, 75, 90, 95]
    )
)

print("\nActual successes:")
print(
    np.percentile(
        default_probabilities[y_test == 1],
        [25, 50, 75, 90, 95]
    )
)

Actual defaults:
[0.18272092 0.34041673 0.60708542 0.7969423  0.86475345]

Actual successes:
[0.06097987 0.11193266 0.19572519 0.33735215 0.48164633]


In [16]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

for threshold in thresholds:

    threshold_predictions = (
        default_probabilities >= threshold
    ).astype(int)

    threshold_predictions = 1 - threshold_predictions

    threshold_default_recall = recall_score(
        y_test,
        threshold_predictions,
        pos_label=0
    )

    threshold_default_precision = precision_score(
        y_test,
        threshold_predictions,
        pos_label=0
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Default recall (Defaulters Caught): {threshold_default_recall:.3f} | "
        f"Default precision (Prediction Accuracy): {threshold_default_precision:.3f}"
    )

Threshold: 0.10 | Default recall (Defaulters Caught): 0.909 | Default precision (Prediction Accuracy): 0.235
Threshold: 0.15 | Default recall (Defaulters Caught): 0.814 | Default precision (Prediction Accuracy): 0.296
Threshold: 0.20 | Default recall (Defaulters Caught): 0.717 | Default precision (Prediction Accuracy): 0.357
Threshold: 0.25 | Default recall (Defaulters Caught): 0.631 | Default precision (Prediction Accuracy): 0.412
Threshold: 0.30 | Default recall (Defaulters Caught): 0.554 | Default precision (Prediction Accuracy): 0.457
Threshold: 0.35 | Default recall (Defaulters Caught): 0.488 | Default precision (Prediction Accuracy): 0.494
Threshold: 0.40 | Default recall (Defaulters Caught): 0.425 | Default precision (Prediction Accuracy): 0.524
Threshold: 0.50 | Default recall (Defaulters Caught): 0.329 | Default precision (Prediction Accuracy): 0.571


In [17]:
for threshold in thresholds:

    threshold_predictions = (
        default_probabilities >= threshold
    ).astype(int)

    threshold_predictions = 1 - threshold_predictions

    predicted_default_count = np.sum(
        threshold_predictions == 0
    )

    predicted_success_count = np.sum(
        threshold_predictions == 1
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Predicted defaults: {predicted_default_count} | "
        f"Predicted successes: {predicted_success_count}"
    )

Threshold: 0.10 | Predicted defaults: 56996 | Predicted successes: 36611
Threshold: 0.15 | Predicted defaults: 40525 | Predicted successes: 53082
Threshold: 0.20 | Predicted defaults: 29638 | Predicted successes: 63969
Threshold: 0.25 | Predicted defaults: 22585 | Predicted successes: 71022
Threshold: 0.30 | Predicted defaults: 17869 | Predicted successes: 75738
Threshold: 0.35 | Predicted defaults: 14555 | Predicted successes: 79052
Threshold: 0.40 | Predicted defaults: 11986 | Predicted successes: 81621
Threshold: 0.50 | Predicted defaults: 8516 | Predicted successes: 85091


In [13]:
from sklearn.metrics import roc_auc_score, average_precision_score

default_target = (y_test == 0).astype(int)

roc_auc = roc_auc_score(
    default_target,
    default_probabilities
)

pr_auc = average_precision_score(
    default_target,
    default_probabilities
)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

ROC-AUC: 0.810101477909096
PR-AUC: 0.472814313225476


## Expanded Model Interpretation
The expanded logistic regression model used borrower financial information—including FICO score, debt-to-income ratio, annual income, revolving credit utilization, account history, public records, mortgage accounts, employment length, and interest rate—to estimate each loan’s probability of default.

The model performed substantially better than the baseline model. Its ROC-AUC was approximately **0.81**, meaning it was generally effective at ranking riskier loans above safer loans. Its PR-AUC was approximately **0.47**, which is meaningful given that defaults represented only about **15.8%** of the test data.

At the standard 0.50 classification threshold, the model achieved:

- Accuracy: **85.5%**
- Default precision: **57.1%**
- Default recall: **32.9%**
- Default F1-score: **41.8%**

The model identified approximately one-third of the loans that actually defaulted. When it classified a loan as a default, it was correct approximately 57% of the time. This was a major improvement over the baseline model, whose default recall was only approximately 0.9%.

The model also separated defaults from successful loans more effectively. The median predicted default probability was approximately **34.0%** for actual defaults, compared with approximately **11.2%** for successful loans. This suggests that the expanded model was better at distinguishing risky loans from safer ones.

The threshold analysis showed the following tradeoff:

| Default Threshold | Default Recall | Default Precision |
|---:|---:|---:|
| 0.10 | 90.9% | 23.5% |
| 0.20 | 71.7% | 35.7% |
| 0.30 | 55.4% | 45.7% |
| 0.40 | 42.5% | 52.4% |
| 0.50 | 32.9% | 57.1% |

A lower threshold catches more actual defaults but incorrectly flags more loans that would have successfully repaid. A higher threshold produces more reliable default predictions but misses more actual defaults.

Overall, the expanded model is a meaningful improvement over the baseline and can rank default risk considerably better. However, it does not yet prove that it has found “invisible prime” borrowers. It currently predicts default risk rather than identifying borrowers whose traditional credit profile understates their actual creditworthiness.

The next stage would be to compare each borrower’s traditional credit indicators, such as FICO score or subgrade, with the model’s predicted default probability. Borrowers with relatively weak traditional credit profiles but unusually low predicted default probabilities could become candidates for further investigation.

Therefore, the main conclusion is that the expanded model is a useful risk-ranking model, but it is not yet a complete invisible-prime investment strategy. Further work is needed to choose a threshold based on expected profit, calibrate the predicted probabilities, test the model on future loans, and evaluate whether the identified borrowers would actually produce profitable lending opportunities.
